# Multimodal Focus Group Discussion: Master’s Thesis Analysis

Upload the original MP4 to retain video + audio evidence. Run with Colab T4 GPU. The notebook produces a clearly labeled machine-assisted, human-validation-required thesis working package.

It samples video frames for observable interaction/context, not identity or emotion inference; speaker labels are intentionally unverified until reviewed.


In [ ]:
# Install free local models and utilities
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -U faster-whisper pandas numpy librosa soundfile python-docx transformers accelerate bitsandbytes qwen-vl-utils
import os, json, subprocess, shutil, re, gc, torch, pandas as pd, numpy as np
from pathlib import Path
from google.colab import files
assert torch.cuda.is_available(), "Set Runtime > Change runtime type > T4 GPU."
ROOT=Path("/content/fgd_thesis"); ROOT.mkdir(exist_ok=True)
print("GPU:",torch.cuda.get_device_name(0))

# Upload the MP4 (preferred: your 1920x1080 video) or an audio file
up=files.upload(); SOURCE=next(iter(up))
suffix=Path(SOURCE).suffix.lower()
assert suffix in {".mp4",".mov",".mkv",".avi",".mp3",".wav",".m4a",".aac",".flac",".ogg",".opus"}
HAS_VIDEO=suffix in {".mp4",".mov",".mkv",".avi"}
print("Source:",SOURCE,"| video:",HAS_VIDEO)

# Master audio, retain brief speech boundaries, and create exact output audio.wav.
# Long silences >1.2 s are shortened to 0.35 s; compact timestamps are therefore not source-video timestamps.
FILTER="highpass=f=70,lowpass=f=7600,afftdn=nr=10:nf=-35:tn=1,acompressor=threshold=-20dB:ratio=2.5:attack=20:release=250,silenceremove=stop_periods=-1:stop_duration=1.2:stop_threshold=-38dB:stop_silence=0.35,loudnorm=I=-16:TP=-1.5:LRA=11"
AUDIO="/content/audio.wav"
subprocess.run(["ffmpeg","-hide_banner","-y","-i",SOURCE,"-vn","-af",FILTER,"-ac","1","-ar","16000","-c:a","pcm_s16le",AUDIO],check=True)
probe=json.loads(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration,size:stream=codec_type,codec_name,width,height,sample_rate,channels","-of","json",SOURCE],text=True))
audio_probe=json.loads(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration,size:stream=sample_rate,channels","-of","json",AUDIO],text=True))
processing={"source":SOURCE,"source_probe":probe,"audio_output":"audio.wav","audio_probe":audio_probe,"audio_processing":"70 Hz high-pass; 7.6 kHz low-pass; gentle denoise; compression; -16 LUFS target; -1.5 dBTP target; silences longer than 1.2 s reduced to 0.35 s.","timestamp_rule":"Transcript times map to compact audio.wav. They do not map directly to original video."}
print(json.dumps(processing,indent=2))

# Transcribe the compact, mastered audio on GPU
from faster_whisper import WhisperModel
whisper=WhisperModel("large-v3",device="cuda",compute_type="float16")
segs,info=whisper.transcribe(AUDIO,beam_size=5,best_of=5,temperature=0,word_timestamps=True,vad_filter=False,condition_on_previous_text=True)
rows=[{"segment_id":i,"compact_start":round(s.start,2),"compact_end":round(s.end,2),"duration":round(s.end-s.start,2),"speaker":"UNVERIFIED_SPEAKER","text":s.text.strip(),"avg_logprob":s.avg_logprob} for i,s in enumerate(segs,1)]
df=pd.DataFrame(rows); assert not df.empty,"No speech was transcribed."
def stamp(t): return f"{int(t//3600):02}:{int(t%3600//60):02}:{int(t%60):02}"
transcript="\n".join(f"[{stamp(r.compact_start)}–{stamp(r.compact_end)}] {r.speaker}: {r.text}" for r in df.itertuples())
(ROOT/"transcript_timestamped.txt").write_text(transcript,encoding="utf-8"); df.to_csv(ROOT/"transcription.csv",index=False,encoding="utf-8-sig")
print("Language:",info.language,round(info.language_probability,3),"segments:",len(df))

# Sample up to 20 spread-out video frames. This records observable interaction/context only—not identity, emotion, or intent.
frames=[]
if HAS_VIDEO:
    frame_dir=ROOT/"video_frames"; frame_dir.mkdir(exist_ok=True)
    duration=float(probe["format"]["duration"])
    times=np.linspace(0,duration,20,endpoint=False)+min(5,duration/40)
    for n,t in enumerate(times,1):
        out=frame_dir/f"frame_{n:02d}_{int(t):04d}s.jpg"
        subprocess.run(["ffmpeg","-hide_banner","-loglevel","error","-y","-ss",str(round(float(t),2)),"-i",SOURCE,"-frames:v","1","-vf","scale='min(960,iw)':-2",str(out)],check=True)
        frames.append({"original_video_time_seconds":round(float(t),2),"file":str(out)})
print("Sampled frames:",len(frames))

# Visual evidence log: Qwen-VL analyses frames conservatively. It never identifies people or infers private traits/emotions.
visual=[]
if frames:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info
    qconf=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16)
    processor=AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
    vl=Qwen2_5_VLForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",quantization_config=qconf,device_map="auto")
    visual_prompt="""This is a sampled frame from a focus-group recording. Describe only plainly observable, research-relevant visual facts: setting, seating/visibility, materials, apparent turn-taking or group interaction, technical/recording issues. Do not identify people, infer demographics, emotions, intent, relationships, or sensitive traits. State uncertainty and write 3–6 short evidence-log bullets."""
    for item in frames:
        msg=[{"role":"user","content":[{"type":"image","image":item["file"]},{"type":"text","text":visual_prompt}]}]
        text=processor.apply_chat_template(msg,tokenize=False,add_generation_prompt=True)
        ii,vi=process_vision_info(msg); inp=processor(text=[text],images=ii,videos=vi,padding=True,return_tensors="pt").to("cuda")
        ids=vl.generate(**inp,max_new_tokens=280,do_sample=False)
        item["observation"]=processor.batch_decode(ids[:,inp.input_ids.shape[1]:],skip_special_tokens=True)[0]
        visual.append(item)
    del vl,processor; gc.collect(); torch.cuda.empty_cache()
(ROOT/"visual_evidence_log.json").write_text(json.dumps(visual,ensure_ascii=False,indent=2),encoding="utf-8")

# Audio/voice technical observations: evidence is acoustic/recording quality only, never speaker identity or health/personality inference.
import librosa
y,sr=librosa.load(AUDIO,sr=16000,mono=True)
voice_note={"compact_duration_seconds":round(len(y)/sr,2),"sample_rate":sr,"channels":1,"mean_rms":round(float(np.sqrt(np.mean(y**2))),5),"estimated_speech_segments":len(df),"interpretation":"Technical recording indicators only. Do not treat these measures as participant traits, emotion, credibility, or diagnosis."}
(ROOT/"audio_voice_technical_notes.json").write_text(json.dumps(voice_note,indent=2),encoding="utf-8")

# Transcript-grounded Q&A and thematic synthesis, using a free local text model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
qconf=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16)
tok=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
llm=AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct",quantization_config=qconf,device_map="auto")
def ask(prompt,n=1700):
    msgs=[{"role":"system","content":"You are a master's-level qualitative research assistant. Use supplied evidence only. Separate transcript evidence, machine observations, and researcher interpretation. Never invent quotations, speakers, identities, demographics, causality, consensus, or visual/emotional facts."},{"role":"user","content":prompt}]
    x=tok.apply_chat_template(msgs,add_generation_prompt=True,return_tensors="pt").to(llm.device)
    z=llm.generate(x,max_new_tokens=n,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(z[0][x.shape[1]:],skip_special_tokens=True)
lines=transcript.splitlines(); chunks=[]; cur=[]; length=0
for line in lines:
    if length+len(line)>8500 and cur: chunks.append("\n".join(cur)); cur=[]; length=0
    cur.append(line); length+=len(line)+1
if cur: chunks.append("\n".join(cur))
part=[ask("Analyse this transcript section. Return: questions asked; verbatim-grounded answers; provisional themes/subthemes; agreement/disagreement; counterexamples; timestamped evidence; data-quality cautions.\n\n"+c) for c in chunks]
visual_text=json.dumps(visual,ensure_ascii=False) if visual else "No video frames were supplied."
report_prompt="""Create a comprehensive, thesis-ready Focus Group Discussion analytic report from the transcript analyses and visual evidence log below. Required headings:
1. Study/Session Metadata (use placeholders where unknown)
2. Data Sources and Processing Audit Trail
3. Transcription and Speaker-Attribution Limitations
4. Audio/Voice Technical Quality Notes
5. Visual Evidence Log (only observable facts; mark machine-generated)
6. Researcher Field Notes and Reflexive Notes Template
7. Question–Answer Matrix (question, evidence-based answer summary, dissent/caveat, timestamps)
8. Thematic Analysis: theme, definition, subthemes, supporting evidence, negative case, interpretation
9. Agreement/Disagreement and Interaction Patterns
10. Preliminary Codebook (code, definition, include/exclude, example, related theme)
11. Findings Narrative
12. Trustworthiness: credibility, dependability, confirmability, transferability
13. Ethics, anonymisation, and data-management notes
14. Master’s Thesis Reporting Checklist
15. Limitations and Required Human Validation.
Never present automated inference as fact. Do not name participants, infer traits, or claim video findings beyond the evidence log.
TRANSCRIPT ANALYSES:
"""+ "\n\n---\n".join(part)+"\n\nVISUAL EVIDENCE LOG:\n"+visual_text+"\n\nAUDIO TECHNICAL NOTE:\n"+json.dumps(voice_note)
report=ask(report_prompt,3000)
(ROOT/"FGD_comprehensive_thesis_report.md").write_text(report,encoding="utf-8")
print(report)

# Deliver a complete thesis working package
from docx import Document
OUT=Path("/content/FGD_THESIS_PACKAGE"); OUT.mkdir(exist_ok=True)
for p in [Path(AUDIO),ROOT/"transcript_timestamped.txt",ROOT/"transcription.csv",ROOT/"visual_evidence_log.json",ROOT/"audio_voice_technical_notes.json",ROOT/"FGD_comprehensive_thesis_report.md"]:
    shutil.copy(p,OUT/p.name)
(OUT/"processing_audit_trail.json").write_text(json.dumps(processing,indent=2),encoding="utf-8")
researcher_template="""# Researcher Notes and Validation Template
Session ID: ____  Date/time: ____  Facilitator: ____  Research question(s): ____

## Context and reflexivity
- Recruitment/context that may shape discussion:
- Facilitator actions or prompts:
- Researcher assumptions/positionality:
- Technical disruptions and their likely effect:

## Validation log
- Transcript passages corrected (timestamp / original / revised / reviewer):
- Speaker labels verified (timestamp / label / evidence):
- Visual observation verified or rejected (video time / observation / reviewer):
- Theme/code changed after review (old / new / rationale):
- Negative cases and dissent retained:

## Analytic memo
- Emerging explanation:
- Competing explanation:
- Evidence needed before thesis claim:
- Quote candidates checked against source:
"""
(OUT/"researcher_notes_template.md").write_text(researcher_template,encoding="utf-8")
doc=Document(); doc.add_heading("Focus Group Discussion — Comprehensive Thesis Analysis",0)
for line in report.splitlines():
    if line.startswith("#"): doc.add_heading(line.lstrip("# ").strip(),min(3,len(line)-len(line.lstrip("#"))))
    elif line.strip(): doc.add_paragraph(line)
doc.add_page_break(); doc.add_heading("Timestamped Transcript",1)
for line in transcript.splitlines(): doc.add_paragraph(line)
doc.save(OUT/"FGD_comprehensive_thesis_report.docx")
zip_path=shutil.make_archive("/content/FGD_THESIS_PACKAGE","zip",OUT)
print("READY:",zip_path); print(*sorted(p.name for p in OUT.iterdir()),sep="\n"); files.download(zip_path)

